# P3 lab: BM25 by hand, in NumPy, and on the climate PDF

Lesson: [`concepts/P3_bm25.md`](../concepts/P3_bm25.md), Part H.1. Concept card: [02 P3](../02_RAG_MASTER_LEARNING_PLAN.md#p3).

**How this lab works (blank-file first).**
- Each task has a **task cell** (what to build), an **empty code cell** for you, and a **CHECK cell** (`# CHECK`, do not edit). Run the CHECK cell after your code. If it fails, read the message and fix your code.
- This notebook has **no solution code**. Write every line yourself. You may look up library docs (the API names are in the lesson, H.1). The reference script in `sources/verification/P3/` is a solution: open it only after an honest attempt and a hint from the tutor (CR-04).
- Where a task says **Predict first**, fill the prediction cell **before** you run anything.
- **No API calls** in this lab (no Groq, no Cohere). Never print values from `.env`.
- Kernel: the project **`.venv` (Python 3.12)**, as in `labs/O2_kernel_check.ipynb`.

## Setup (given)
Run this cell. It names the input file, gives the toy corpus from the lesson (Part F), and defines `_fp`, a fingerprint that the CHECK cells use so the expected answers stay hidden until you have predicted.

In [ ]:
import hashlib, json, math
import numpy as np
PDF = r"C:\Yash\New-Work\Book RAG\upstream\RAG_Techniques\data\Understanding_Climate_Change.pdf"

def _fp(x):
    """Fingerprint of a value. The CHECK cells compare fingerprints, not visible answers."""
    return hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()[:16]

TOY = [
    "BPA regulation (2019): the EU limits BPA in food-contact plastics.",
    "Climate policy and plastics: a review of chemical safety regulation.",
    "The 2019 BPA ban covers baby bottles; BPA, BPS and BPF are bisphenols.",
    "Climate change raises sea levels and flood risks for coastal cities.",
    "Greenhouse gas emissions and climate targets under the Paris Agreement.",
    "Plastics manufacturing adds climate emissions."
]   # toy corpus, made up for P3 (lesson Part F)
print(len(TOY), "toy documents")

## Task 1: two tokenizers

Write two functions, each taking a string and returning a `list[str]`:
- `tok_split(text)`: exactly what the repo does (`fusion_retrieval.ipynb` cell 14): split on whitespace, nothing else.
- `tok_norm(text)`: lower-case the text, then keep runs of letters and digits as tokens. Everything else is a separator, **except** that a `.` or `,` **between two digits** stays inside the token, so `1.5` and `650,000` survive whole.

Test them on this sentence (given in the CHECK): `"Limit warming to 1.5 degrees; CO2, BPA (IPCC) & a 2019 ban of 650,000 t."`

**Predict first** (in the cell below): how many tokens will `tok_split` give for that sentence, and how many will `tok_norm` give?

In [ ]:
my_prediction_n_split = None   # replace None with a number, BEFORE you run your tokenizers
my_prediction_n_norm = None

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p3_measure.py).

In [ ]:
# CHECK Task 1 (do not edit)
SAMPLE = "Limit warming to 1.5 degrees; CO2, BPA (IPCC) & a 2019 ban of 650,000 t."
assert my_prediction_n_split is not None and my_prediction_n_norm is not None, "write both predictions first"
assert tok_split("a  b\tc\n") == ["a", "b", "c"], "tok_split must split on any whitespace, like str.split()"
assert tok_norm("Hello, World!") == ["hello", "world"], "tok_norm must lower-case and drop punctuation"
assert tok_norm("1.5 and 650,000.") == ["1.5", "and", "650,000"], "a . or , between digits stays inside the token; a final . does not"
assert _fp(tok_split(SAMPLE)) == "491ee1d3438c3de0", f"tok_split(SAMPLE) is not right yet: {tok_split(SAMPLE)}"
assert _fp(tok_norm(SAMPLE)) == "8c67c7c2854bf76a", f"tok_norm(SAMPLE) is not right yet: {tok_norm(SAMPLE)}"
print("Task 1 OK.", len(tok_split(SAMPLE)), "split tokens (you predicted", my_prediction_n_split, ");",
      len(tok_norm(SAMPLE)), "norm tokens (you predicted", my_prediction_n_norm, ")")
print("split:", tok_split(SAMPLE))
print("norm: ", tok_norm(SAMPLE))

**Which split() tokens can never match a user who types `IPCC` or `co2`?** One sentence.

*Template:* "The tokens ____ cannot match, because ____."

*Your answer:*

## Task 2: one BM25 term score by hand

Tokenize `TOY` with your `tok_norm` and print the six token lists. Then, **by reading the printed lists** (count with your eyes, not with code), fill the five counts for the term `bpa` in document `d0` (the first document):
- `N`: number of documents; `n_bpa`: documents that contain `bpa`; `f_bpa_d0`: times `bpa` occurs in d0;
- `dl_d0`: tokens in d0; `avgdl`: average tokens per document (a float).

Then compute, **writing the formula yourself** with `math.log` (no library BM25), with k1 = 1.5 and b = 0.75:
- `hand_idf` = ln((N − n + 0.5) / (n + 0.5)) (this is the IDF that `rank_bm25` uses; lesson D.3);
- `hand_K` = k1 · (1 − b + b · dl / avgdl);
- `hand_tfpart` = f · (k1 + 1) / (f + K);
- `hand_score` = `hand_idf` · `hand_tfpart`.

**Predict first:** roughly what will `hand_score` be (between 0 and 3)?

In [ ]:
my_prediction_score_bpa_d0 = None   # replace None with a number, BEFORE you compute

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p3_measure.py).

In [ ]:
# CHECK Task 2 (do not edit)
from rank_bm25 import BM25Okapi
assert my_prediction_score_bpa_d0 is not None, "write your prediction first"
assert _fp([N, n_bpa, f_bpa_d0, dl_d0, float(avgdl)]) == "090ac4bcb5e1fd63", \
    "one of N, n_bpa, f_bpa_d0, dl_d0, avgdl is off: recount from the printed token lists"
_lib = BM25Okapi([tok_norm(d) for d in TOY]).get_scores(["bpa"])[0]
assert abs(hand_score - hand_idf * hand_tfpart) < 1e-12, "hand_score must be hand_idf * hand_tfpart"
assert abs(hand_score - _lib) < 1e-6, "hand_score does not match rank_bm25 to 1e-6: check idf, K and the tf part step by step"
print(f"Task 2 OK. idf={hand_idf:.6f} K={hand_K:.6f} tf part={hand_tfpart:.6f} score={hand_score:.6f};"
      f" you predicted {my_prediction_score_bpa_d0}")

## Task 3: BM25 in NumPy (no `rank_bm25`)

Write `bm25_scores(corpus_tokens, query_tokens, k1=1.5, b=0.75, epsilon=0.25)` that returns a NumPy array with one score per document, **identical** to `rank_bm25.BM25Okapi(corpus_tokens, k1=k1, b=b, epsilon=epsilon).get_scores(query_tokens)`.

Build it from: a vocabulary; a term-frequency matrix (documents × terms); document lengths and `avgdl`; document frequencies `n`; the IDF from Task 2; then the library's two special rules (lesson D.3):
1. a term whose IDF is **strictly negative** gets the floor `epsilon × (mean IDF over all terms, computed before flooring)`;
2. a query term that is not in the vocabulary adds 0, and a query term that is repeated is added again.

Do **not** import `rank_bm25` inside your function. The CHECK compares your function with the library on the toy corpus, on edge cases and on 200 random corpora.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p3_measure.py).

In [ ]:
# CHECK Task 3 (do not edit)
import inspect, random
from rank_bm25 import BM25Okapi
assert "rank_bm25" not in inspect.getsource(bm25_scores) and "BM25Okapi" not in inspect.getsource(bm25_scores), \
    "write BM25 yourself: no rank_bm25 inside bm25_scores"
def _same(corpus, query, **kw):
    mine = np.asarray(bm25_scores(corpus, query, **kw), dtype=float)
    lib = BM25Okapi(corpus, **kw).get_scores(query)
    return mine.shape == lib.shape and np.allclose(mine, lib, atol=1e-9, rtol=0)
_T = [tok_norm(d) for d in TOY]
assert _same(_T, ["bpa", "regulation", "2019"]), "toy corpus, query 'bpa regulation 2019': scores differ"
assert _same(_T, ["climate"]), "a term in MORE than half the documents: did you apply the epsilon floor?"
assert _same(_T, ["plastics"]), "a term in EXACTLY half the documents: idf is 0, and 0 is not negative"
assert _same(_T, ["bpa", "bpa"]), "a repeated query term must be added twice"
assert _same(_T, ["ipcc"]), "a query term not in the vocabulary must add 0 (no error)"
assert _same([["bpa", "ban"], ["climate", "policy"]], ["bpa"]), "two-document corpus case"
assert _same([["a", "b", "c"], ["a", "b"], ["a", "b"]], ["a"]), "negative mean idf case: the floor itself can be negative"
assert _same(_T, ["bpa", "climate"], k1=3.0, b=0.0) and _same(_T, ["bpa", "eu"], k1=0.5, b=1.0), "k1 and b must be used"
rnd = random.Random(0)
for _ in range(200):
    V = [f"w{i}" for i in range(rnd.randint(2, 12))]
    corpus = [[rnd.choice(V) for _ in range(rnd.randint(1, 15))] for _ in range(rnd.randint(1, 9))]
    query = [rnd.choice(V + ["oov"]) for _ in range(rnd.randint(1, 4))]
    kw = dict(k1=rnd.choice([0.5, 0.9, 1.2, 1.5, 2.0]), b=rnd.choice([0.0, 0.5, 0.75, 1.0]), epsilon=rnd.choice([0.0, 0.25, 0.5]))
    assert _same(corpus, query, **kw), f"random corpus differs: corpus={corpus} query={query} {kw}"
_mine_d0 = float(np.asarray(bm25_scores(_T, ["bpa"]))[0])
assert abs(_mine_d0 - hand_score) < 1e-6, "your code and your hand computation (Task 2) disagree by more than 1e-6"
print(f"Task 3 OK. Hand {hand_score:.6f} vs code {_mine_d0:.6f}: |diff| = {abs(_mine_d0 - hand_score):.1e} (02 card evidence: <= 1e-6)")

## Task 4: saturation (k1) and length normalisation (b)

Write `tf_part(f, dl_over_avgdl, k1=1.5, b=0.75)` = f · (k1 + 1) / (f + k1 · (1 − b + b · dl_over_avgdl)).

**Predict first:**
1. At k1 = 1.5 and dl = avgdl, what value does `tf_part` approach as f grows without limit?
2. With b = 0, does a one-mention term score more in a short document (dl = 0.5·avgdl) or in a long one (dl = 4·avgdl)?

In [ ]:
my_prediction_tf_limit = None      # a number
my_prediction_b0 = None            # "short", "long" or "same"

Now build:
- `sat_table`: a dict `{"1": ..., "2": ..., "5": ..., "10": ..., "100": ..., "1000": ...}` (keys are **strings**) with `tf_part(f, 1.0)` rounded to 4 decimals;
- `b0_row`: a list of `tf_part(1, r, b=0.0)` rounded to 4 decimals, for r = 0.5, 1.0, 2.0, 4.0.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p3_measure.py).

In [ ]:
# CHECK Task 4 (do not edit)
assert my_prediction_tf_limit is not None and my_prediction_b0 in ("short", "long", "same"), "write both predictions first"
assert abs(tf_part(1, 1.0) - 1.0) < 1e-12, "at f=1 and dl=avgdl the tf part is exactly 1 (for any k1): check the formula"
assert _fp(round(float(tf_part(1, 2.0)), 4)) == "59b30db8264a8fe3", "tf_part(1, 2.0) is wrong: dl_over_avgdl multiplies b (dl / avgdl, not avgdl / dl)"
assert _fp({k: round(float(v), 4) for k, v in sat_table.items()}) == "29f693b9526d25b3", "sat_table is not right yet (keys are strings; round to 4)"
assert _fp([round(float(v), 4) for v in b0_row]) == "e9dbaf68429c1a0f", "b0_row is not right yet"
print("Task 4 OK.", sat_table)
print("b = 0 row:", b0_row, "| you predicted", my_prediction_b0, "| your tf limit prediction:", my_prediction_tf_limit)

**Evidence for the 02 card (one sentence each, no notes).**

*tf → ∞:* "As tf grows, the term's score approaches ____ × ____, because ____."

*b → 0:* "With b = 0, the score ____ on document length, so ____."

*Your answers:*

## Task 5: what if k1 doubles, or b halves? (CR-08)

**Predict first:** (a) a term occurs **twice** in a document of average length. k1 doubles from 1.5 to 3.0. Does its tf part go up, down or stay the same? (b) a term occurs **once** in a document **twice** the average length. b halves from 0.75 to 0.375. Up, down or the same?

In [ ]:
my_prediction_k1_double = None   # "up", "down" or "same"
my_prediction_b_half = None      # "up", "down" or "same"

Build `k1_double = [before, after]` and `b_half = [before, after]` with your `tf_part`, each rounded to 4 decimals.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p3_measure.py).

In [ ]:
# CHECK Task 5 (do not edit)
assert my_prediction_k1_double in ("up", "down", "same") and my_prediction_b_half in ("up", "down", "same"), "write both predictions first"
assert _fp([round(float(x), 4) for x in k1_double]) == "9a1de284e9dd76ee", "k1_double is not right yet (f=2, dl=avgdl, k1 1.5 then 3.0)"
assert _fp([round(float(x), 4) for x in b_half]) == "0a4f050b1db29fb1", "b_half is not right yet (f=1, dl=2*avgdl, b 0.75 then 0.375)"
print(f"Task 5 OK. k1 1.5->3.0: {k1_double} (you said {my_prediction_k1_double}); b 0.75->0.375: {b_half} (you said {my_prediction_b_half})")

## Task 6: exact terms on the climate PDF

Load the PDF as in the P1 lab (PyMuPDF `page.get_text()`, pages joined with `"\n"`). Cut **plain** character windows of 1,000 with overlap 200: `windows = [full[i:i+1000] for i in range(0, len(full), 800)]`.

Write `best_target_rank(scores, target_ids)`: the 1-based rank of the best-scoring window among `target_ids`, counted as 1 + the number of windows with a **strictly higher** score. If that best target score is 0, return `None` (BM25 did not find it at all).

For each query in `QUERIES` (given below), the target windows are the windows whose raw text contains the target string. Build `ranks = {query: [rank with tok_split, rank with tok_norm]}`. Use `BM25Okapi` (default k1, b) or your own `bm25_scores`; they give the same scores.

In [ ]:
QUERIES = [("IPCC", "IPCC"), ("What does BECCS stand for?", "BECCS"), ("methane CH4", "CH4"),
           ("Kyoto Protocol 1997", "1997"), ("limit warming to 1.5 degrees", "1.5")]   # (query, target string)
my_prediction_split_fails = None   # list the queries you expect tok_split to MISS completely (rank None), BEFORE running

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p3_measure.py).

In [ ]:
# CHECK Task 6 (do not edit)
assert my_prediction_split_fails is not None, "write your prediction first"
assert len(windows) == 91 and len(windows[0]) == 1000, "expected 91 windows of 1,000 characters (the last is shorter)"
assert best_target_rank(np.array([0.0, 0.0, 0.0]), [1]) is None, "a zero target score means not found: return None"
assert best_target_rank(np.array([3.0, 1.0, 3.0, 2.0]), [3]) == 3 and best_target_rank(np.array([3.0, 1.0, 3.0]), [1, 2]) == 1, \
    "rank = 1 + number of windows with a strictly higher score than the best target"
_r = {q: [None if x is None else int(x) for x in v] for q, v in ranks.items()}
assert set(_r) == {q for q, _ in QUERIES}, "one entry per query"
assert _fp(_r) == "17671005c943ff08", f"ranks are not right yet: {_r}"
print("Task 6 OK.")
for q, (a, b) in _r.items():
    print(f"  {q!r:34s} split -> {a}   norm -> {b}")
print("You predicted split would miss:", my_prediction_split_fails)

**Diagnose (one sentence each).** Why does `tok_split` miss `IPCC` completely? And is this a retrieval failure or a generation failure (CR-11)?

*Template:* "The text holds the token ____, so the query token ____ never matches. It is a ____ failure, because ____."

*Your answer:*

## Task 7: debugging: a query that matches nothing still returns k chunks

The repo picks the top k with `np.argsort(scores)[::-1][:k]` (`helper_functions.py` L268). When every score is 0 (the `IPCC` query under `split()`), that still returns k windows, chosen only by position.

Write `safe_top_k(scores, k)` that returns a list of `int` window indices: the windows with score **> 0**, highest first, ties broken by the **lower** index, at most k of them. A query that matches nothing returns `[]`.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p3_measure.py).

In [ ]:
# CHECK Task 7 (do not edit)
assert safe_top_k(np.zeros(91), 5) == [], "all-zero scores must return an empty list"
assert safe_top_k(np.array([0.0, 2.0, 1.0, 2.0, 0.0]), 5) == [1, 3, 2], "only positive scores, highest first, ties by lower index"
assert safe_top_k(np.array([0.5, 0.0, 0.7]), 1) == [2], "at most k results"
assert all(isinstance(i, int) for i in safe_top_k(np.array([1.0, 2.0]), 2)), "return plain int indices"
print("Task 7 OK. Repo-style top 5 for an all-zero vector:", np.argsort(np.zeros(91))[::-1][:5].tolist(), "| yours:", safe_top_k(np.zeros(91), 5))

## Wrap-up: evidence for the log

Answer each in one line, no notes.

1. *Hand vs code:* "My hand score for `bpa` in d0 was ____ and my code gave ____; the difference was ____."
2. *tf → ∞:* "____."
3. *b → 0:* "____."
4. *Tokenization:* "`split()` missed ____ because ____; `tok_norm` fixed it but would break ____."

> Tell the tutor when you are done. The tutor grades `coding_independence_0_3` (Tasks 1, 3, 6) and `debugging_0_3` (Task 7, and any failed CHECK you fixed yourself).